# Module 3b — WIPO Merge (Patents & Trademarks)

**What we're adding:** `patent_applications` and `trademark_applications` — total counts by
applicant's origin (resident + abroad combined), from your manual WIPO IP Statistics export.
Real coverage is roughly 2015-2024 for most countries, sparser before that — matches what
WIPO's own reporting depth looks like, not a loading error.

**The one real gotcha in this data:** WIPO's `Origin (Code)` column uses **ISO alpha-2**
codes (`AF`, `AL`, `DZ`), but everything else in this pipeline uses **ISO alpha-3**
(`AFG`, `ALB`, `DZA`). Merging without converting would silently produce 0% coverage — no
error, just nothing matching. This notebook fixes that explicitly before touching the merge.

Drop both exported CSVs into `data/raw/wipo/` (any filenames) before running this.

In [25]:
import os
from pathlib import Path

while not (Path("config").exists() and Path("scripts").exists()):
    os.chdir("..")
    if Path.cwd() == Path.cwd().parent:
        raise RuntimeError("Could not locate project root")

print("Working directory set to:", os.getcwd())


Working directory set to: c:\Users\AJAY\Desktop\startup_implementation


## Stage 1 — load both files, auto-detect which is which

Rather than assume filenames, read the first line of every CSV in `data/raw/wipo/` — WIPO's
own export tells us directly ("Intellectual property right :Patent" vs ":Trademark") so we
don't have to guess based on what you happened to name the file.

In [26]:
import pandas as pd
from pathlib import Path

WIPO_RAW_DIR = Path("data/raw/wipo")
wipo_files = sorted(WIPO_RAW_DIR.glob("*.csv"))
print(f"{len(wipo_files)} files found in {WIPO_RAW_DIR}:")

file_map = {}  # "Patent" / "Trademark" -> Path

for f in wipo_files:
    with open(f, encoding="utf-8-sig") as fh:
        preview_lines = [fh.readline().strip() for _ in range(3)]
    print(f" - {f.name}")

    # Some WIPO exports include metadata lines identifying the IP type; others (a plain
    # CSV download) start directly at the "Origin," header with no such line. Check both
    # the content and the filename, so this works either way.
    haystack = " ".join(preview_lines) + " " + f.name
    if "patent" in haystack.lower() and "trademark" not in f.name.lower():
        file_map["Patent"] = f
    elif "trademark" in haystack.lower():
        file_map["Trademark"] = f

print("\nDetected:", {k: v.name for k, v in file_map.items()})


for label in ["Patent", "Trademark"]:
    if label not in file_map:
        print(f"WARNING: no {label} file detected — {label.lower()}_applications will NOT be added. Check the filename/content above.")

2 files found in data\raw\wipo:
 - patent_2005_2024.csv
 - trademark_2005_2024.csv

Detected: {'Patent': 'patent_2005_2024.csv', 'Trademark': 'trademark_2005_2024.csv'}


## Stage 2 — parse WIPO's export format

WIPO's CSVs have a few metadata lines before the real header (which starts with "Origin,").
Rather than hardcode `skiprows=3` and hope it's always exactly 3 lines, find the header row
by searching for it — robust against WIPO changing their export format slightly between
releases.

In [27]:
def load_wipo_csv(path):
    with open(path, encoding="utf-8-sig") as fh:
        lines = fh.readlines()

    header_idx = next(i for i, line in enumerate(lines) if line.startswith("Origin,"))

    # index_col=False is required: WIPO's export has a trailing comma on every data
    # row, which otherwise makes pandas silently use "Origin" as a row index and
    # shift every other column one position left (Type becomes 2005's value, years
    # get dropped, etc.) with no error. Do not remove this.
    df = pd.read_csv(path, skiprows=header_idx, encoding="utf-8-sig", index_col=False, keep_default_na=False, na_values=[""])

    # Drop the trailing empty column from WIPO's trailing comma
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    return df

patent_raw = load_wipo_csv(file_map["Patent"]) if "Patent" in file_map else None
trademark_raw = load_wipo_csv(file_map["Trademark"]) if "Trademark" in file_map else None

for label, df in [("Patent", patent_raw), ("Trademark", trademark_raw)]:
    if df is not None:
        print(f"\n{label}: {df.shape}")
        print(df.columns.tolist())
        display(df.head(3))



Patent: (930, 24)
['Origin', 'Origin (Code)', 'Office', 'Type', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


,Origin,Origin (Code),Office,Type,2005,2006,2007,2008,2009,2010,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AF,Total,Abroad,1.0,NaN,2.0,2.0,2.0,6.0,...,5.0,11.0,29.0,11.0,9.0,8.0,16.0,17.0,7.0,6.0
1,Afghanistan,AF,Total,Abroad (equivalent count),1.0,NaN,2.0,2.0,2.0,6.0,...,5.0,11.0,29.0,11.0,9.0,8.0,16.0,17.0,7.0,6.0
2,Albania,AL,Total,Total,1.0,NaN,NaN,NaN,4.0,NaN,...,16.0,37.0,18.0,21.0,12.0,NaN,32.0,28.0,28.0,NaN



Trademark: (969, 24)
['Origin', 'Origin (Code)', 'Office', 'Type', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


,Origin,Origin (Code),Office,Type,2005,2006,2007,2008,2009,2010,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AF,Total,Abroad,10.0,20.0,19.0,41.0,19.0,13.0,...,30.0,93.0,136.0,90.0,88.0,140.0,1016.0,135.0,72.0,50.0
1,Afghanistan,AF,Total,Abroad (equivalent count),10.0,20.0,19.0,93.0,19.0,91.0,...,30.0,147.0,163.0,90.0,88.0,140.0,1016.0,165.0,100.0,50.0
2,Albania,AL,Total,Total,178.0,193.0,165.0,226.0,234.0,321.0,...,586.0,732.0,742.0,1051.0,858.0,921.0,1015.0,803.0,1144.0,1189.0


## Stage 3 — build the alpha-2 -> alpha-3 country code mapping

Same World Bank country metadata endpoint Module 2 used for aggregate filtering — this time
we keep `iso2Code` alongside the alpha-3 `id`, specifically to translate WIPO's codes.

In [28]:
import requests

def fetch_alpha2_to_alpha3(cache_path="data/raw/wipo_country_code_map.csv"):
    cache = Path(cache_path)
    cache.parent.mkdir(parents=True, exist_ok=True)
    try:
        url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        _, records = r.json()
        meta = pd.DataFrame(records, dtype=str)
        meta = meta[["id", "iso2Code", "name"]].rename(columns={"id": "alpha3"})
        meta = meta[meta["iso2Code"].notna() & (meta["iso2Code"] != "")]
        meta.to_csv(cache, index=False)
        print(f"Fetched live mapping: {len(meta)} entities")
        return meta
    except requests.exceptions.RequestException as e:
        if cache.exists():
            print(f"Live fetch failed ({e.__class__.__name__}) — using cached copy from {cache}")
            return pd.read_csv(cache)
        raise RuntimeError(
            "Could not reach api.worldbank.org and no cached mapping exists. "
            "Run this notebook with internet access at least once."
        ) from e

code_map = fetch_alpha2_to_alpha3()
alpha2_to_alpha3 = dict(zip(code_map["iso2Code"], code_map["alpha3"]))


Fetched live mapping: 295 entities


## Stage 4 — reshape to long format and merge

Filter to `Type == "Total"` (resident + abroad combined, non-equivalent count — matches what
you selected in the export UI), melt the wide year columns into rows, convert country codes,
and left-join onto `world_bank_wide.csv`.

In [29]:
def wipo_to_long(df, value_name):
    if df is None:
        return None

    year_cols = [c for c in df.columns if c.strip().isdigit()]

    filtered = df[df["Type"] == "Total"].copy()

    long = filtered.melt(
        id_vars=["Origin", "Origin (Code)"],
        value_vars=year_cols,
        var_name="year",
        value_name=value_name,
    )

    long["year"] = long["year"].astype(int)
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")
    long = long.dropna(subset=[value_name])

    long["country_code"] = long["Origin (Code)"].map(alpha2_to_alpha3)

    unmatched = long[long["country_code"].isna()]["Origin"].unique()
    if len(unmatched):
        print(f"{value_name}: {len(unmatched)} origin(s) had no alpha-3 match "
              f"(likely non-sovereign territories/regional aggregates in WIPO's list) — dropped:")
        print(sorted(unmatched)[:20])

    long = long.dropna(subset=["country_code"])

    return long[["country_code", "year", value_name]]

patent_long = wipo_to_long(patent_raw, "patent_applications")
trademark_long = wipo_to_long(trademark_raw, "trademark_applications")

for label, df in [("patent_applications", patent_long), ("trademark_applications", trademark_long)]:
    if df is not None:
        print(f"\n{label}: {len(df)} rows, {df.country_code.nunique()} countries, "
              f"years {df.year.min()}-{df.year.max()}")


patent_applications: 1 origin(s) had no alpha-3 match (likely non-sovereign territories/regional aggregates in WIPO's list) — dropped:
['Least Developed Countries']
trademark_applications: 3 origin(s) had no alpha-3 match (likely non-sovereign territories/regional aggregates in WIPO's list) — dropped:
['Least Developed Countries', 'Netherlands Antilles', 'Yugoslavia']

patent_applications: 2621 rows, 177 countries, years 2005-2024

trademark_applications: 2812 rows, 186 countries, years 2005-2024


In [30]:
wide_df = pd.read_csv("data/cleaned/world_bank_wide.csv")
print("Before merge:", wide_df.shape)

merged = wide_df
for df in [patent_long, trademark_long]:
    if df is not None:
        merged = merged.merge(df, on=["country_code", "year"], how="left")

print("After merge:", merged.shape)
for col in ["patent_applications", "trademark_applications"]:
    if col in merged.columns:
        cov = merged[col].notna().mean()
        print(f"{col}: {merged[col].notna().sum()} non-null ({cov:.1%} coverage)")

merged.to_csv("data/cleaned/world_bank_wide.csv", index=False)
print("\nSaved back to data/cleaned/world_bank_wide.csv")


Before merge: (4557, 41)
After merge: (4557, 43)

Saved back to data/cleaned/world_bank_wide.csv


## Next

With World Bank + WGI + Entrepreneurship + UNESCO + WIPO all merged, the last optional piece
is the supplementary startup columns (GEM TEA rate, StartupBlink/GSER rank) — otherwise we're
ready to copy this file to `data/master/startup_master_dataset.csv` as the real final
deliverable.